<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c02-header.png" alt="Nextia Learning · Python for Practical AI Engineering" width="100%">

# Solution: Exceptions and logging

**[Exceptions and logging](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/)** · Python for Practical AI Engineering · Module 5, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** separate expected input errors from programming defects, catch only the errors that you expect, end with a clear message and exit code, and use logging instead of `print` for what the program does.

| | |
|---|---|
| **Time** | About 45 minutes |
| **Needs** | An internet connection: the setup installs httpx and downloads the data files. No account. |
| **You should know** | The two fixes of [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/): `find_problems` rejects a value that is not a dictionary, and `read_rows` reads CSV with `utf-8-sig`. Exit codes, from [Files and terminal](https://learning.nextia-ai.com/courses/python/m01/files-and-terminal/#exit-codes-did-the-command-work). |
| **You do not need** | Your local project and its environment. The setup makes the same `ticket-cleaner` folder here. |
| **Tested** | Python 3.14.6 with httpx 0.28.1, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every example, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C02/M05-L02-exceptions-and-logging/exceptions-and-logging-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `httpx` 0.28.1 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `httpx 0.28.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check httpx==0.28.1
import httpx; print("httpx", httpx.__version__)

## Setup: make the project folder

Run the next cell. It makes a folder `ticket-cleaner`, laid out like your project from Module 1, and works in it. It writes the files that you made in the earlier lessons (the code is in the cell). It needs no internet connection.

It also downloads the course's ticket files (`tickets.csv`, `tickets.json`, `tickets-excel.csv`, under 4 KB together) into `data/`, and checks each file's checksum, as you did with `curl` in [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/#get-the-data-file). This part needs an internet connection and no account. If it shows `Could not download`, check your connection and run it again (in Kaggle, turn on **Internet** in the notebook settings). If it shows `wrong checksum`, delete the folder `ticket-cleaner` (in Colab: the **Files** panel on the left), then run it again. The data is made up for this course; it has no real people.

You should see `Ready: ticket-cleaner, 15 files`. If you run the cell again, it puts every file back as at the start of this lesson: use this to start again.

The files are the code of these lessons, as you have it at the end of each one: [Decisions and repetition](https://learning.nextia-ai.com/courses/python/m02/decisions-and-repetition/) (count_tickets.py), [Function contracts](https://learning.nextia-ai.com/courses/python/m03/function-contracts/) (count_tickets.py as functions), [Modules and imports](https://learning.nextia-ai.com/courses/python/m03/modules-and-imports/) (the package `ticket_cleaner` with `report.py`), [Structured records](https://learning.nextia-ai.com/courses/python/m03/structured-records/) (`records.py` and `parsing.py`), [Reading and writing files](https://learning.nextia-ai.com/courses/python/m04/reading-and-writing-files/) (`files.py`, `split_records` and `build_summary`), [Basic HTTP client use](https://learning.nextia-ai.com/courses/python/m04/basic-http-client-use/) (`remote.py` and `try_mock.py`), [Configuration and secrets](https://learning.nextia-ai.com/courses/python/m04/configuration-and-secrets/) (`config.py` and `.env.example`), [Read failures](https://learning.nextia-ai.com/courses/python/m05/read-failures/) (the two fixes in `parsing.py` and `files.py`).

In [ ]:
# Setup: make the ticket-cleaner folder as at the start of this lesson. Download the data files.
import hashlib, os, shutil, subprocess, sys, urllib.request
from pathlib import Path

if Path.cwd().name == "ticket-cleaner":  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-cleaner").resolve()
for item in PROJECT.glob("*"):  # start again: remove everything except the data files
    if item.name != "data":
        shutil.rmtree(item) if item.is_dir() else item.unlink()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/"
DATA = {
    "tickets.csv": "82e3fb4fa6d94a47b9a7b05d940a088596363e4dd11e89bf82ed486bd2325a1a",
    "tickets.json": "a801f505e3d91ff79cefee9c72e53b5f48e983ee152305b7f04e3ac882dd9946",
    "tickets-excel.csv": "cf0fbbce193bb1a595ce9207cc3f004aeedb118008e4ffa35585fda2f1167a95",
}

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in DATA.items():
    path = PROJECT / "data" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-cleaner and run again.")


FILES = {
    'count_tickets.py': '''\
from pathlib import Path

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> None:
    settings = load_settings()
    print(f"Source: {settings.source}")
    rows = load_rows(settings)
    tickets, rejected = split_records(rows)
    for record in rejected:
        print(f"Row {record.row} rejected: {'; '.join(record.problems)}")

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    print(f"Report: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()
''',
    'requirements.txt': '''\
httpx==0.28.1
''',
    'ticket_cleaner/__init__.py': '''\
"""Clean support-ticket records and summarise them."""
''',
    'ticket_cleaner/report.py': '''\
from ticket_cleaner.records import Rejected, Ticket


def filter_by_status(tickets: list[Ticket], status: str = "open") -> list[Ticket]:
    """Return the tickets that have the given status."""
    return [ticket for ticket in tickets if ticket.status == status]


def count_by_category(tickets: list[Ticket]) -> dict[str, int]:
    """Return the number of tickets in each category, sorted by category."""
    counts = {}
    for ticket in tickets:
        counts[ticket.category] = counts.get(ticket.category, 0) + 1
    return dict(sorted(counts.items()))


def average_priority(tickets: list[Ticket]) -> float | None:
    """Return the average priority, or None when there are no tickets."""
    if not tickets:
        return None
    return round(sum(ticket.priority for ticket in tickets) / len(tickets), 1)


def build_summary(tickets: list[Ticket], rejected: list[Rejected], status: str) -> dict:
    """Return the report as a dictionary that can be saved as JSON."""
    selected = filter_by_status(tickets, status)
    return {
        "status": status,
        "valid_records": len(tickets),
        "selected": len(selected),
        "by_category": count_by_category(selected),
        "average_priority": average_priority(selected),
        "rejected": [{"row": r.row, "problems": r.problems} for r in rejected],
    }
''',
    'ticket_cleaner/records.py': '''\
from dataclasses import dataclass

STATUSES = {"open", "pending", "closed"}
PRIORITIES = {1, 2, 3}
REQUIRED_FIELDS = ("id", "status", "category", "priority")


@dataclass(frozen=True)
class Ticket:
    """One valid support ticket."""

    id: str
    status: str
    category: str
    priority: int

    def is_urgent(self) -> bool:
        """Return True for priority 1, the most urgent level."""
        return self.priority == 1


@dataclass(frozen=True)
class Rejected:
    """An input record that is not a valid ticket, and why."""

    row: int
    problems: list[str]
''',
    'ticket_cleaner/parsing.py': '''\
from ticket_cleaner.records import PRIORITIES, REQUIRED_FIELDS, STATUSES, Rejected, Ticket


def clean_text(value: object) -> str:
    """Return the value as text with no outer spaces, in small letters."""
    if value is None:
        return ""
    return str(value).strip().lower()


def find_problems(raw: dict) -> list[str]:
    """Return a list of problems with one raw record. An empty list means valid."""
    if not isinstance(raw, dict):
        return ["not a record"]
    problems = []
    for field in REQUIRED_FIELDS:
        if clean_text(raw.get(field)) == "":
            problems.append(f"missing {field}")

    status = clean_text(raw.get("status"))
    if status and status not in STATUSES:
        problems.append(f"unknown status {status!r}")

    priority = clean_text(raw.get("priority"))
    if priority and not (priority.isdigit() and int(priority) in PRIORITIES):
        problems.append(f"priority must be 1, 2 or 3, not {priority!r}")
    return problems


def to_ticket(raw: dict) -> Ticket:
    """Make a Ticket from a raw record that has no problems."""
    return Ticket(
        id=clean_text(raw["id"]).upper(),
        status=clean_text(raw["status"]),
        category=clean_text(raw["category"]),
        priority=int(clean_text(raw["priority"])),
    )


def split_records(rows: list[dict]) -> tuple[list[Ticket], list[Rejected]]:
    """Separate valid tickets from rejected records. Nothing is dropped silently."""
    tickets = []
    rejected = []
    seen_ids = set()
    for row_number, raw in enumerate(rows, start=1):
        problems = find_problems(raw)
        if not problems:
            ticket = to_ticket(raw)
            if ticket.id in seen_ids:
                problems.append(f"duplicate id {ticket.id}")
        if problems:
            rejected.append(Rejected(row=row_number, problems=problems))
        else:
            tickets.append(ticket)
            seen_ids.add(ticket.id)
    return tickets, rejected
''',
    'ticket_cleaner/files.py': '''\
import csv
import json
import os
from pathlib import Path


def read_rows(path: Path) -> list[dict]:
    """Read raw records from a .csv or .json file."""
    if path.suffix == ".csv":
        with open(path, encoding="utf-8-sig", newline="") as file:
            return list(csv.DictReader(file))
    if path.suffix == ".json":
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    raise ValueError(f"cannot read {path.name}: use a .csv or .json file")


def write_json(data: dict, path: Path) -> None:
    """Save data as JSON. Write a temporary file first, so a failure never leaves half a file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
        file.write("\\n")
    os.replace(temporary, path)
''',
    'ticket_cleaner/remote.py': '''\
import httpx


def fetch_rows(
    url: str, token: str | None = None, client: httpx.Client | None = None
) -> list[dict]:
    """Download raw records as JSON from a ticket service."""
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    client = client or httpx.Client(timeout=10.0)
    with client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()
''',
    'try_mock.py': '''\
import httpx

from ticket_cleaner.remote import fetch_rows


def answer_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json=[{"id": "T-2001", "status": "open"}])


def answer_unavailable(request: httpx.Request) -> httpx.Response:
    return httpx.Response(503, text="Service Unavailable")


def main() -> None:
    client = httpx.Client(transport=httpx.MockTransport(answer_ok))
    print(fetch_rows("https://tickets.example/api/tickets", client=client))

    client = httpx.Client(transport=httpx.MockTransport(answer_unavailable))
    fetch_rows("https://tickets.example/api/tickets", client=client)


if __name__ == "__main__":
    main()
''',
    'ticket_cleaner/config.py': '''\
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
    )
''',
    '.env.example': '''\
# Settings for ticket-cleaner. Set them as environment variables.
# TICKETS_SOURCE: a .csv or .json file, or "remote". Default: data/tickets.csv
TICKETS_SOURCE=data/tickets.csv
# TICKETS_URL: the web service for "remote". Default: the course's public file.
TICKETS_URL=
# TICKETS_API_TOKEN: secret. Never write a real token in this file.
TICKETS_API_TOKEN=
''',
    'data/broken.json': '''\
[
  {"id": "T-3001", "status": "open", "category": "billing", "priority": 2},
  null,
  {"id": "T-3003", "status": "closed", "category": "login", "priority": 3}
]
''',
}
for name, text in FILES.items():
    (PROJECT / name).parent.mkdir(parents=True, exist_ok=True)
    (PROJECT / name).write_text(text, encoding="utf-8")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
sys.path.insert(0, str(PROJECT))
sys.dont_write_bytecode = True  # always read the newest version of each file
count = sum(1 for path in PROJECT.rglob("*") if path.is_file())
print("Ready: ticket-cleaner" + (f", {count} file{'s' if count > 1 else ''}" if count else ""))

## Setup: the helpers

The next cell defines small functions that this notebook uses. You do not need to read them, but you can.

- `check("""assert ...""")` runs the `assert` lines of a check cell. It prints `Check passed.`, or `Not yet:` and what is different. It does not show the answer.
- `run("count_tickets.py")` runs a file of the project with Python, as `python count_tickets.py` does in your terminal, and shows its output and its exit code. `run("-m", "pytest")` is `python -m pytest`. `env={"NAME": "value"}` sets an environment variable for that one run.
- The check functions of this lesson's tasks. They print `Check passed.` or `Not yet:` and what is different.

Run the cell. It prints nothing.

In [ ]:
# Helpers: run the project, and check an answer without showing it.
import os, sys

def check(asserts):
    """Run the assert lines of a check. Say what is different, never the answer."""
    try:
        exec(asserts, globals())
    except AssertionError as error:
        print(f"Not yet: {error}")
    except NameError as error:
        print(f"Not yet: {error}. Run your cell above first.")
    except Exception as error:
        print(f"Not yet: the check stopped with {type(error).__name__}: {error}")
    else:
        print("Check passed.")

import subprocess, sys

def environment(env=None):
    """The environment variables of a run: the notebook's, plus `env` (None
    removes one). No colours, so the output reads as plain text."""
    variables = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1",
                 "PYTHON_COLORS": "0", "NO_COLOR": "1", "PY_COLORS": "0"}
    variables.pop("FORCE_COLOR", None)
    for name, value in (env or {}).items():
        if value is None:
            variables.pop(name, None)
        else:
            variables[name] = value
    return variables

def run_quiet(*args, env=None):
    """Run `python <args>` in the project folder. Return the result
    (.returncode, .stdout, .stderr)."""
    return subprocess.run([sys.executable, *args], env=environment(env), text=True, capture_output=True)

def run(*args, env=None, streams="both"):
    """Run `python <args>` in the project folder, like the terminal, and show
    the output and the exit code. streams="stdout" or "stderr" shows one only."""
    if streams == "both":  # the two streams together, in the order they were written
        result = subprocess.run([sys.executable, *args], env=environment(env), text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output = result.stdout
    else:
        result = run_quiet(*args, env=env)
        output = result.stdout if streams == "stdout" else result.stderr
    if output:
        print(output, end="" if output.endswith("\n") else "\n")
    print(f"(exit code {result.returncode})")

def check_predictions():
    """Compare the predicted exit codes with the real ones, without showing them."""
    real = {"cut": 1, "bug": 1, "error_level": 0}
    if any(predictions.get(name) is None for name in real):
        print("Not yet: write a number for each of the three situations.")
        return
    wrong = [name for name in real if predictions[name] != real[name]]
    if wrong:
        print(f"Not yet: your prediction for {', '.join(wrong)} is different. Run the cells below, and find out why.")
    else:
        print("Check passed.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two kinds of error

An **exception** is Python's way to report an error. When code **raises** one, Python goes back up through the callers. If no caller handles it, the program stops with the traceback. Ask: **whose fault is it, and can the program do something useful about it?**

- **Expected errors** come from the world outside: the file does not exist, the server is down, the JSON is cut short. They can happen in correct code. Handle them: stop with a clear message and exit code `1`.
- **Programming defects** (bugs) come from the program itself: a typing mistake in a field name, a wrong argument. Do not handle them: let the traceback show the line, and fix the code.

## 2. try and except

A `try` block runs code that can raise an exception. An `except` block says which exception types to handle. A **different** exception type is not caught: it goes up to the caller as before.

> **Predict.** Which of the three values does the `except` catch? Then run the cell. (The outer `try` only shows the error that the inner `except` does not catch, so that the notebook continues.)

In [ ]:
def to_priority(text: str) -> int:
    return int(text)


try:
    for text in ["2", "high", None]:
        try:
            print(text, "->", to_priority(text))
        except ValueError as error:
            print(text, "-> rejected:", error)
except Exception as error:
    print("Not caught:", type(error).__name__, error)

> **Check.** You should see `2 -> 2`, `high -> rejected: invalid literal for int() with base 10: 'high'`, and `Not caught: TypeError int() argument must be a string, a bytes-like object or a real number, not 'NoneType'`. Is a `None` here an expected error or a defect? Think about where the value comes from.

Exception types form families. A more general type covers the specific types under it. Run the cell.

In [ ]:
import json
import httpx

print(issubclass(FileNotFoundError, OSError))
print(issubclass(PermissionError, OSError))
print(issubclass(json.JSONDecodeError, ValueError))
print(issubclass(httpx.HTTPStatusError, httpx.HTTPError))
print(issubclass(httpx.ConnectTimeout, httpx.HTTPError))
print(issubclass(AttributeError, ValueError))

> **Check.** You should see `True` five times, then `False`: `except (OSError, ValueError, httpx.HTTPError)` catches the input errors, and not an `AttributeError` from a typing mistake.

## 3. Catch at the edge, and log

The `try` goes at the **edge** of the program, in `main()`, which knows the whole situation. `read_rows` and `fetch_rows` raise, and let the caller decide. `main()` returns the exit code, and `raise SystemExit(main())` passes it to Python.

A **log** is better than `print` for messages about what the program **does**. Each message has a **level**: `DEBUG`, `INFO`, `WARNING` or `ERROR`. You choose the lowest level to show when the program starts. Log messages go to **standard error** (stderr), and `print` goes to **standard output** (stdout).

Run the next two cells. The first saves a small script that logs two messages. The second runs it.

In [ ]:
%%writefile log_demo.py
import logging

logger = logging.getLogger("ticket_cleaner")
logging.basicConfig(level="WARNING", format="%(levelname)s %(name)s: %(message)s")

logger.info("reading tickets from %s", "data/tickets.csv")   # hidden at WARNING
logger.warning("row %d rejected: %s", 7, "unknown status 'opne'")

In [ ]:
run("log_demo.py")

> **Check.** You should see only `WARNING ticket_cleaner: row 7 rejected: unknown status 'opne'`. The `INFO` message is hidden at the level `WARNING`. The values after the message replace `%s` (text) and `%d` (a whole number), in order. A good log message says which record and why.

## 4. Worked example: count_tickets.py fails well

`config.py` gets a `log_level` setting from `LOG_LEVEL`, so the level changes without a change to the code. `count_tickets.py` catches the three families of expected errors in one place, logs the rejected records, and returns an exit code. The `try` block holds one line: the line that touches the outside world. Run the two cells.

In [ ]:
%%writefile ticket_cleaner/config.py
import os
from dataclasses import dataclass, field

DEFAULT_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C02/data/tickets.json"


@dataclass(frozen=True)
class Settings:
    source: str
    tickets_url: str
    api_token: str | None = field(repr=False)
    log_level: str


def load_settings() -> Settings:
    """Read the settings from environment variables, with safe defaults."""
    return Settings(
        source=os.environ.get("TICKETS_SOURCE", "data/tickets.csv"),
        tickets_url=os.environ.get("TICKETS_URL", DEFAULT_URL),
        api_token=os.environ.get("TICKETS_API_TOKEN") or None,
        log_level=os.environ.get("LOG_LEVEL", "WARNING").upper(),
    )

In [ ]:
%%writefile count_tickets.py
import logging
from pathlib import Path

import httpx

from ticket_cleaner.config import Settings, load_settings
from ticket_cleaner.files import read_rows, write_json
from ticket_cleaner.parsing import split_records
from ticket_cleaner.remote import fetch_rows
from ticket_cleaner.report import build_summary

OUTPUT_FILE = Path("reports/summary.json")

logger = logging.getLogger("ticket_cleaner")


def load_rows(settings: Settings) -> list[dict]:
    """Return raw records from a file, or from the web service for "remote"."""
    if settings.source == "remote":
        return fetch_rows(settings.tickets_url, token=settings.api_token)
    return read_rows(Path(settings.source))


def main() -> int:
    settings = load_settings()
    logging.basicConfig(
        level=settings.log_level, format="%(levelname)s %(name)s: %(message)s"
    )
    logger.info("reading tickets from %s", settings.source)

    try:
        rows = load_rows(settings)
    except (OSError, ValueError, httpx.HTTPError) as error:
        logger.error("cannot read the tickets from %s: %s", settings.source, error)
        return 1

    tickets, rejected = split_records(rows)
    for record in rejected:
        logger.warning("row %d rejected: %s", record.row, "; ".join(record.problems))

    summary = build_summary(tickets, rejected, status="open")
    write_json(summary, OUTPUT_FILE)
    logger.info("wrote %s", OUTPUT_FILE)
    print(f"{len(rows)} rows: {len(tickets)} valid, {len(rejected)} rejected.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 5. Guided practice: make it fail well

Run the cell. It runs the program three times: with the defaults, with `LOG_LEVEL=info`, and with a file that does not exist.

In [ ]:
run("count_tickets.py")
run("count_tickets.py", env={"LOG_LEVEL": "info"})
run("count_tickets.py", env={"TICKETS_SOURCE": "data/missing.csv"})

> **Check.**
>
> - The first run shows `15 rows: 9 valid, 6 rejected.` and six `WARNING ticket_cleaner: row … rejected: …` lines, and `(exit code 0)`.
> - The second also shows `INFO ticket_cleaner: reading tickets from data/tickets.csv` and `INFO ticket_cleaner: wrote reports/summary.json`.
> - The third shows one line, `ERROR ticket_cleaner: cannot read the tickets from data/missing.csv: [Errno 2] No such file or directory: 'data/missing.csv'`, no traceback, and `(exit code 1)`.

See the two output streams. In a terminal, `2>/dev/null` hides stderr and `>/dev/null` hides stdout. Here, `streams="stdout"` shows only the result, and `streams="stderr"` shows only the log. Run the cell.

In [ ]:
run("count_tickets.py", streams="stdout")
run("count_tickets.py", streams="stderr")

> **Check.** The first run shows only `15 rows: 9 valid, 6 rejected.` The second shows only the six warnings. Another program, or a file, can take the result without the log messages.

## 6. Your turn: predict, then run

> **Your turn.** For each situation, predict the exit code (`0` or `1`). Write your three numbers in the next cell, and run it and the check cell. Then run the three cells after it, which make each situation and undo it.
>
> 1. `cut`: a JSON file that is cut short, `data/cut.json` with the text `[{"id": 1` (no end), as `TICKETS_SOURCE`.
> 2. `bug`: in `report.py`, `ticket.category` in `count_by_category` changed to `ticket.categry`.
> 3. `error_level`: `LOG_LEVEL=error` with the normal data file.

In [ ]:
predictions = {
    "cut": 1,
    "bug": 1,
    "error_level": 0,
}

In [ ]:
check_predictions()

In [ ]:
from pathlib import Path

Path("data/cut.json").write_text('[{"id": 1', encoding="utf-8")
run("count_tickets.py", env={"TICKETS_SOURCE": "data/cut.json"})
Path("data/cut.json").unlink()

In [ ]:
report = Path("ticket_cleaner/report.py")
correct = report.read_text(encoding="utf-8")
report.write_text(correct.replace("ticket.category]", "ticket.categry]"), encoding="utf-8")
run("count_tickets.py")
report.write_text(correct, encoding="utf-8")  # undo the bug
print("report.py is correct again.")

In [ ]:
run("count_tickets.py", env={"LOG_LEVEL": "error"})

> **Check.**
>
> 1. `ERROR ticket_cleaner: cannot read the tickets from data/cut.json: Expecting ',' delimiter: line 1 column 10 (char 9)`, exit code `1`. `json.JSONDecodeError` is a `ValueError`, so the `except` catches it: a file can be damaged at any time.
> 2. A full traceback that ends with `AttributeError: 'Ticket' object has no attribute 'categry'. Did you mean: 'category'?`, exit code `1`. The `except` does not catch it, because `build_summary` is outside the `try` block. That is correct: it is a defect, and the traceback shows the line to fix.
> 3. Only `15 rows: 9 valid, 6 rejected.`, exit code `0`. The warnings are hidden, because `WARNING` is below `ERROR`. `print` is not part of the log.

## 7. Failure case: a catch-all except that hides a bug

Tomás puts the whole program in one `try` block, with `except Exception:`. It catches almost every exception, including defects.

> **Predict.** What does the user see when `build_summary` has the typing mistake? Then run the cell.

In [ ]:
def build_summary_with_bug():
    raise AttributeError("'Ticket' object has no attribute 'categry'")

try:
    rows = ["…"]
    summary = build_summary_with_bug()
except Exception:
    print("ERROR ticket_cleaner: cannot read the tickets")

> **Check.** You should see `ERROR ticket_cleaner: cannot read the tickets`. The message blames the input, and the traceback, which would show the real line, is gone.

**Fix:** put only the lines that touch the outside world in the `try` block, and name the exception types that you expect. If you must catch everything at the very top of a program, use `logger.exception("...")`, which writes the full traceback into the log, and end with exit code `1`.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/). They count toward your certificate when you are signed in and enrolled. On your computer, run the program with `echo $?` (or `$LASTEXITCODE`) to see each exit code, as in the guided practice of the lesson.

In this lesson you learned to handle expected errors and let defects crash, to catch specific types at the edge of the program, to return `1` for an error, and to log messages about the run to stderr while the result goes to stdout.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Focused tests](https://learning.nextia-ai.com/courses/python/m05/focused-tests/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/python/m05/exceptions-and-logging/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The tickets in this course are made up; they have no real people.